# Código Task 2

## Librerias

In [ ]:
"""
Entorno: GridWorld 4x4 simple
- El agente parte de (0,0) y busca llegar a (3,3)
- Recompensa: -1 por cada paso, +10 al llegar a la meta, -5 si cae en celda trampa
- El agente sigue una política aleatoria (no aprende todavía — solo observamos el ciclo)
"""

import numpy as np
import random

## Constantes

In [ ]:
GRID_SIZE = 4
GOAL_STATE = (3, 3)          # Estado terminal con recompensa positiva
TRAP_STATES = {(1, 1), (2, 3)}  # Estados trampa
# TRAP_STATES = {(1, 1), (2, 3), (0, 3)}  # Task 2, inciso 2
MAX_STEPS = 50               # Horizonte máximo del episodio
GAMMA = 0.95                 # Factor de descuento

# Acciones
# Espacio de acciones discreto: 4 direcciones posibles
ACTIONS = {
    0: (-1, 0),   # Arriba
    1: (1, 0),    # Abajo
    2: (0, -1),   # Izquierda
    3: (0, 1),    # Derecha
}
ACTION_NAMES = {0: "↑", 1: "↓", 2: "←", 3: "→"}

## Clases para Entorno y Agente

In [ ]:
# Entorno
class GridWorldEnv:
    """
    Representa el entorno (Environment) en el vocabulario de RL.

    Responsabilidades:
    - Mantener el estado actual del mundo
    - Transicionar al nuevo estado dado una acción
    - Emitir la recompensa R_{t+1} después de cada transición
    """

    def reset(self):
        """Inicializa el entorno y devuelve el estado inicial S_0."""
        self.state = (0, 0)
        self.done = False
        return self.state

    def step(self, action):
        """
        Ejecuta un paso del ciclo agente-entorno.

        Recibe: A_t (action)
        Devuelve: S_{t+1}, R_{t+1}, done

        Esta función implementa la función de transición P(S_{t+1} | S_t, A_t)
        y la función de recompensa R(S_t, A_t, S_{t+1}).
        """
        if self.done:
            raise RuntimeError("El episodio ya terminó. Llama reset().")

        dr, dc = ACTIONS[action]
        r, c = self.state

        # La transición respeta los bordes del grid (entorno determinista)
        new_r = max(0, min(GRID_SIZE - 1, r + dr))
        new_c = max(0, min(GRID_SIZE - 1, c + dc))
        next_state = (new_r, new_c)

        # Función de recompensa: señal de diseño del ingeniero
        if next_state == GOAL_STATE:
            reward = +10.0    # Meta alcanzada
            self.done = True
        elif next_state in TRAP_STATES:
            reward = -5.0     # Trampa activada
            self.done = True
        else:
            reward = -1.0     # Costo por paso (incentiva llegar rápido)

        self.state = next_state
        return next_state, reward, self.done


# Agente con politica aleatoria
class RandomAgent:
    """
    Agente con política estocástica uniforme.

    Política π(a|s) = 1/|A| para todo a — distribución uniforme.
    No aprende — sirve para ilustrar el ciclo de interacción.
    """

    def __init__(self, action_space_size):
        self.n_actions = action_space_size

    def select_action(self, state):
        """
        Muestrea una acción de la política estocástica uniforme.

        En un agente que aprende (Q-learning, PPO, etc.), esta función
        consultaría una tabla Q o una red neuronal.
        """
        return random.randint(0, self.n_actions - 1)

## Funciones

In [ ]:
# Ciclo de agnte entorno
def run_episode(env, agent, gamma=GAMMA, max_steps=MAX_STEPS, verbose=True):
    """
    Ejecuta un episodio completo y calcula el retorno G_0.

    Implementa directamente el ciclo de la Diapositiva 3:
        S_0 → A_0 → R_1, S_1 → A_1 → R_2, S_2 → ...
    """
    state = env.reset()                    # S_0
    trajectory = []                        # Almacena (S_t, A_t, R_{t+1})
    total_reward = 0.0

    if verbose:
        print(f"\n{'═'*50}")
        print(f"INICIO DEL EPISODIO  Estado inicial: {state}")
        print(f"{'─'*50}")

    for t in range(max_steps):
        action = agent.select_action(state)                    # A_t ~ π(·|S_t)
        next_state, reward, done = env.step(action)            # Transición

        trajectory.append((state, action, reward, next_state))
        total_reward += reward

        if verbose:
            status = "✓ META" if next_state == GOAL_STATE \
                     else "✗ TRAMPA" if next_state in TRAP_STATES \
                     else ""
            print(f"  t={t:2d}  S={state}  A={ACTION_NAMES[action]}  "
                  f"R={reward:+.1f}  S'={next_state}  {status}")

        state = next_state

        if done:
            break

    # Calcula el retorno descontado G_0 = Σ γ^k * R_{k+1}
    # Procesa la trayectoria en reversa para acumulación eficiente
    G = 0.0
    returns = []
    for _, _, r, _ in reversed(trajectory):
        G = r + gamma * G                  # G_t = R_{t+1} + γ * G_{t+1}
        returns.insert(0, G)

    if verbose:
        print(f"{'─'*50}")
        print(f"  Pasos totales        : {len(trajectory)}")
        print(f"  Recompensa acumulada : {total_reward:+.1f}")
        print(f"  Retorno G_0 (γ={gamma}): {returns[0]:.4f}")
        print(f"{'═'*50}\n")

    return trajectory, returns


# Experimentar con multiples episodios
def run_experiment(n_episodes=5):
    """
    Corre múltiples episodios para ilustrar la variabilidad de una política
    estocástica — con la misma política, los resultados varían por azar.
    """
    env = GridWorldEnv()
    agent = RandomAgent(action_space_size=len(ACTIONS))

    print("SIMULACIÓN: Ciclo Agente-Entorno en GridWorld 4×4")
    print("Política: aleatoria uniforme (sin aprendizaje)")
    print(f"Factor de descuento γ = {GAMMA}")

    all_returns = []
    for ep in range(n_episodes):
        print(f"\n▶ Episodio {ep + 1}/{n_episodes}")
        _, returns = run_episode(env, agent, verbose=True)
        all_returns.append(returns[0])

    print("\n" + "═"*50)
    print("RESUMEN DEL EXPERIMENTO")
    print(f"  Retorno promedio G_0 : {np.mean(all_returns):.4f}")
    print(f"  Retorno máximo       : {max(all_returns):.4f}")
    print(f"  Retorno mínimo       : {min(all_returns):.4f}")
    print("═"*50)
    print("\nObservación: la variabilidad entre episodios ilustra por qué")
    print("el objetivo de RL es el RETORNO ESPERADO, no un retorno fijo.")

## Ejecución

In [ ]:
if __name__ == "__main__":
    random.seed(42)
    np.random.seed(42)
    run_experiment(n_episodes=5)

# Task 2

## Inciso 1

Según las instrucciones, se modificó la constante **GAMMA**, corriendo el script una vez con **GAMMA = 0.5** y otra vez con **GAMMA = 0.99**, manteniendo la misma seed para que los episodios generados por la política aleatoria fueran comparables entre corridas.

```python
GAMMA = 0.5   
GAMMA = 0.99  
```

Copie el output para comparar resultados en ambas simulaciones.

## Output — GAMMA = 0.5

```
▶ Episodio 1/5
  Pasos totales        : 9
  Recompensa acumulada : -13.0
  Retorno G_0 (γ=0.5): -2.0117

▶ Episodio 2/5
  Pasos totales        : 22
  Recompensa acumulada : -26.0
  Retorno G_0 (γ=0.5): -2.0000

▶ Episodio 3/5
  Pasos totales        : 50
  Recompensa acumulada : -39.0
  Retorno G_0 (γ=0.5): -2.0000

▶ Episodio 4/5
  Pasos totales        : 17
  Recompensa acumulada : -21.0
  Retorno G_0 (γ=0.5): -2.0000

▶ Episodio 5/5
  Pasos totales        : 13
  Recompensa acumulada : -17.0
  Retorno G_0 (γ=0.5): -2.0007

══════════════════════════════════════════════════
RESUMEN DEL EXPERIMENTO
  Retorno promedio G_0 : -2.0025
  Retorno máximo       : -2.0000
  Retorno mínimo       : -2.0117
══════════════════════════════════════════════════
```

## Output — GAMMA = 0.99

```
▶ Episodio 1/5
  Pasos totales        : 9
  Recompensa acumulada : -13.0
  Retorno G_0 (γ=0.99): -12.3393

▶ Episodio 2/5
  Pasos totales        : 22
  Recompensa acumulada : -26.0
  Retorno G_0 (γ=0.99): -23.0759

▶ Episodio 3/5
  Pasos totales        : 50
  Recompensa acumulada : -39.0
  Retorno G_0 (γ=0.99): -32.7771

▶ Episodio 4/5
  Pasos totales        : 17
  Recompensa acumulada : -21.0
  Retorno G_0 (γ=0.99): -19.1115

▶ Episodio 5/5
  Pasos totales        : 13
  Recompensa acumulada : -17.0
  Retorno G_0 (γ=0.99): -15.7934

══════════════════════════════════════════════════
RESUMEN DEL EXPERIMENTO
  Retorno promedio G_0 : -20.6194
  Retorno máximo       : -12.3393
  Retorno mínimo       : -32.7771
══════════════════════════════════════════════════
```

## Tabla comparativa

| γ | Retorno promedio G₀ | Máximo | Mínimo |
|---|---|---|---|
| 0.5 | -2.0025 | -2.0000 | -2.0117 |
| 0.95 (original) | -13.5606 | -10.0487 | -17.5702 |
| 0.99 | -20.6194 | -12.3393 | -32.7771 |

---

Como usamos la misma semilla, los episodios son exactamente iguales en las tres corridas; lo único que cambia es γ. Con γ=0.5 el descuento castiga tan rápido los pasos futuros que `G_0` casi solo refleja los primeros pasos, por eso todos los episodios dan un retorno parecido (≈-2.0) sin importar si duraron 9 o 50 pasos. Con γ=0.99 casi no hay descuento, entonces los episodios más largos acumulan un `G_0` mucho más negativo (hasta -32.78), porque el agente "siente" el costo de todo el camino.

## Inciso 2

Agregue un nuevo TRAP_STATE en la posición (0, 3). ¿Cómo afecta esto la distribución de retornos?

### Output

```bash
SIMULACIÓN: Ciclo Agente-Entorno en GridWorld 4×4
Política: aleatoria uniforme (sin aprendizaje)
Factor de descuento γ = 0.95

▶ Episodio 1/5

══════════════════════════════════════════════════
INICIO DEL EPISODIO  Estado inicial: (0, 0)
──────────────────────────────────────────────────
...
──────────────────────────────────────────────────
  Pasos totales        : 9
  Recompensa acumulada : -13.0
  Retorno G_0 (γ=0.95): -10.0487
══════════════════════════════════════════════════


▶ Episodio 2/5

══════════════════════════════════════════════════
INICIO DEL EPISODIO  Estado inicial: (0, 0)
──────────────────────────────────────────────────
...
──────────────────────────────────────────────────
  Pasos totales        : 22
  Recompensa acumulada : -26.0
  Retorno G_0 (γ=0.95): -14.8916
══════════════════════════════════════════════════


▶ Episodio 3/5

══════════════════════════════════════════════════
INICIO DEL EPISODIO  Estado inicial: (0, 0)
──────────────────────────────────────────────────
...
──────────────────────────────────────────────────
  Pasos totales        : 50
  Recompensa acumulada : -39.0
  Retorno G_0 (γ=0.95): -17.5702
══════════════════════════════════════════════════


▶ Episodio 4/5

══════════════════════════════════════════════════
INICIO DEL EPISODIO  Estado inicial: (0, 0)
──────────────────────────────────────────────────
...
──────────────────────────────────────────────────
  Pasos totales        : 17
  Recompensa acumulada : -21.0
  Retorno G_0 (γ=0.95): -13.3981
══════════════════════════════════════════════════


▶ Episodio 5/5

══════════════════════════════════════════════════
INICIO DEL EPISODIO  Estado inicial: (0, 0)
──────────────────────────────────────────────────
  t= 0  S=(0, 0)  A=↑  R=-1.0  S'=(0, 0)
  t= 1  S=(0, 0)  A=→  R=-1.0  S'=(0, 1)
  t= 2  S=(0, 1)  A=→  R=-1.0  S'=(0, 2)
  t= 3  S=(0, 2)  A=→  R=-5.0  S'=(0, 3)  ✗ TRAMPA
──────────────────────────────────────────────────
  Pasos totales        : 4
  Recompensa acumulada : -8.0
  Retorno G_0 (γ=0.95): -7.1394
══════════════════════════════════════════════════


══════════════════════════════════════════════════
RESUMEN DEL EXPERIMENTO
  Retorno promedio G_0 : -12.6096
  Retorno máximo       : -7.1394
  Retorno mínimo       : -17.5702
══════════════════════════════════════════════════
```

### Respuesta

Agregar la trampa en `(0, 3)` introduce un nuevo estado terminal (absorbente) con recompensa -5 cerca del inicio, lo que bajo la misma politica aleatoria uniforme desplaza la distribucion de retornos descontados G_0 hacia valores menos negativos: cambia el Episodio 5, que ahora termina en la nueva trampa al paso 4 con G_0 = -7.14 en vez de -11.89, subiendo el promedio de -13.56 a -12.61 y el maximo de -10.05 a -7.14 (el minimo no cambia). El estado terminal cerca del arranque trunca las trayectorias largas y les evita acumular el costo de -1 por paso, lo que ilustra la hipotesis de recompensa: la recompensa es una senal de diseno y modificar un solo estado reconfigura toda la distribucion, recordando que el objetivo de RL es el retorno esperado (E[G_0]) sobre muchos episodios, no un retorno fijo.